# 01 · When to fine-tune — and how to prepare the data

After this notebook you can answer "should we fine-tune?" the way a senior engineer does (usually: not yet),
pick the right method from the fine-tuning menu, and turn a pile of raw examples into a clean, de-duplicated,
leak-free, chat-formatted training set.

**Time:** ~25 min · **Runs:** offline on CPU in < 1 min (downloads only the SmolLM2 tokenizer, already cached) ·
**Needs:** nothing — this is the first notebook of the fine-tuning section.

## Why this matters in interviews

- "Should we fine-tune for this?" is the most common fine-tuning question, and the most common wrong answer is
  "yes, so the model knows our documents". Knowing the decision order (prompt → RAG → tune) is the signal.
- Interviewers probe the **data**, not the optimizer: de-duplication, label balance, train/eval leakage, and how
  many examples you need. Bad data is the usual reason a fine-tune loses to the base model.
- Chat templates and special tokens are the most common *silent* self-hosting bug — no error, just worse answers.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tn11` | When should you fine-tune versus use RAG versus just prompt better? |
| `tn40` | Your model needs to answer questions about documents that change weekly. Fine-tune or retrieve? Defend it. |
| `tn03` | What is fine-tuning, and what are its main types? |
| `tn26` | What is the difference between prompt tuning, prefix tuning and LoRA? |
| `tn37` | What is the difference between SFT and preference tuning, in terms of what each can teach? |
| `tn39` | How do you decide between fine-tuning a small model and prompting a large one? |
| `tn13` | How much data do you need to fine-tune, and what does good data look like? |
| `tn25` | How would you build a fine-tuning dataset from production traffic? |
| `tn24` | What is a chat template and why does it matter when self-hosting? |
| `pr26` | What is the difference between few-shot examples and fine-tuning data? |

In [ ]:
import json, re, hashlib, tempfile, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

random.seed(0)
rng = np.random.default_rng(0)
pd.set_option("display.width", 140, "display.max_colwidth", 70)

def draw_pipeline(steps, title=None):
    """Boxes joined by arrows, left to right."""
    fig, ax = plt.subplots(figsize=(min(2.3 * len(steps), 11), 1.7))
    for i, s in enumerate(steps):
        x = i * 2.3
        ax.add_patch(FancyBboxPatch((x, 0), 1.9, 1, boxstyle="round,pad=0.06", fc="#e7f0ff", ec="#3b5bdb"))
        ax.text(x + 0.95, 0.5, s, ha="center", va="center", fontsize=9)
        if i < len(steps) - 1:
            ax.annotate("", xy=(x + 2.28, 0.5), xytext=(x + 1.98, 0.5), arrowprops=dict(arrowstyle="->", lw=1.6))
    ax.set_xlim(-0.2, len(steps) * 2.3); ax.set_ylim(-0.25, 1.25); ax.axis("off")
    if title: ax.set_title(title)
    plt.show()

## 1. Prompt vs RAG vs fine-tune

**In plain English:** if the model gets the *facts* wrong, give it the facts (retrieval). If it has the facts but gets
the *format, tone or behaviour* wrong, that is what tuning fixes. And if you have not yet tried writing better
instructions with two or three examples, do that first — it costs an afternoon and it is reversible.

The rule that survives every follow-up: **fine-tuning teaches form, retrieval supplies facts.** Knowledge comes almost
entirely from pretraining; SFT and preference tuning change *how* a model behaves far more than *what* it knows.

| Question to ask | Prompt + few-shot | RAG | Fine-tune (LoRA / full) |
|---|---|---|---|
| The model lacks **knowledge** of our data | no | **yes** | poor — facts blur into weights, no citations |
| The facts **change** weekly (freshness) | – | **yes** — re-index in seconds | no — retrain every week, forever |
| Must **cite**, **delete** or **permission-filter** facts | – | **yes** | no — you cannot delete a fact from weights |
| **Format / style / tone / behaviour** is wrong | often enough | no | **yes** — the thing tuning is for |
| A long few-shot block is sent on **every** call | costs tokens each call | – | **yes** — bakes it in, shorter prompts |
| Added **latency** | more prompt tokens | + retrieval (tens–hundreds of ms) | none once merged |
| **Up-front cost** | ~0 | build an index | data + training + an eval suite + an owner |
| **Time to change** | minutes | minutes | days |

They compose: the standard production shape is a model tuned for *your format*, fed by retrieval that supplies
*your facts*.

In [ ]:
draw_pipeline(["1. Prompt +\nfew-shot", "2. RAG\n(facts)", "3. LoRA SFT\n(behaviour)", "4. Preference\ntuning (DPO)",
               "5. Full FT /\ncont. pretrain"],
              title="Decision order: cheapest, most reversible first — move right only with evidence")

The same decision order as code. Each scenario is a real interview prompt; the flags are the questions you would ask
the interviewer back. The *order* of the `if`s is the point: facts are checked before behaviour, and prompting is
checked before anything.

In [ ]:
def recommend(s: dict) -> tuple[str, str]:
    """Decision order from tn11: prompt first, retrieval for facts, tuning for behaviour."""
    if not s.get("prompt_tried"):
        return "prompt + few-shot", "free, instant, reversible — most 'we need a fine-tune' tickets die here"
    if s.get("needs_facts"):
        return "RAG", "facts must be fresh / cited / deletable / permission-filtered"
    if s.get("new_language_or_domain") and s.get("n_examples", 0) >= 50_000:
        return "continued pretraining + full FT", "far from the pretraining distribution, lots of data"
    if s.get("better_easier_to_judge_than_write"):
        return "preference tuning (DPO)", "you can rank answers but not write the perfect one"
    if s.get("high_volume_narrow_task"):
        return "fine-tune / distil a small model", "per-call cost and latency dominate at this volume"
    if s.get("behaviour_wrong") and s.get("n_examples", 0) >= 100:
        return "LoRA SFT", "consistent format/style problem and enough examples of 'right'"
    return "prompt + few-shot (keep iterating)", "no evidence that training would help yet"

scenarios = {
    "Answer questions about docs that change weekly":        dict(prompt_tried=True, needs_facts=True),
    "Outputs too long; no system prompt written yet":         dict(prompt_tried=False),
    "Wrong JSON layout despite a good prompt; 800 good rows": dict(prompt_tried=True, behaviour_wrong=True, n_examples=800),
    "Tone: easy to pick the better reply, hard to write it": dict(prompt_tried=True, better_easier_to_judge_than_write=True),
    "Clinical notes in Icelandic, 2B tokens + 80k pairs":     dict(prompt_tried=True, new_language_or_domain=True, n_examples=80_000),
    "Classify tickets into 12 intents at 5M calls/day":       dict(prompt_tried=True, high_volume_narrow_task=True),
    "Answers must cite the exact policy clause":              dict(prompt_tried=True, needs_facts=True),
    "Style is a bit off; we have 30 examples":                dict(prompt_tried=True, behaviour_wrong=True, n_examples=30),
}
for scenario, flags_ in scenarios.items():
    choice, why = recommend(flags_)
    print(f"{scenario:56s} -> {choice}\n{'':56s}    ({why})")
assert recommend(scenarios["Answer questions about docs that change weekly"])[0] == "RAG"

**Read the output:** the two "facts" scenarios go to RAG even though fine-tuning was possible — freshness, citations
and deletion are properties of an index, not of weights (`tn40`). The 30-example style problem stays with prompting:
30 rows is a few-shot block, not a training set (`pr26`: start few-shot, log what works, fine-tune when the example
block becomes the biggest part of your prompt).

### Small fine-tuned model vs prompting a large one (`tn39`)

This is a volume calculation. Prompting a big model has no fixed cost but a higher price per call, forever. A small
fine-tuned model is pennies per call plus a fixed monthly cost: training, an eval suite, a retraining trigger and a
person who owns it. The numbers below are **illustrative assumptions** — plug in your own.

In [ ]:
big_per_call   = 500 * 2.00 / 1e6 + 200 * 8.00 / 1e6    # 500 in / 200 out tokens at $2 / $8 per 1M (assumed)
small_per_call = 150 * 0.10 / 1e6 + 200 * 0.40 / 1e6    # shorter prompt (no few-shot) on a small model (assumed)
small_fixed_per_month = 5_000 / 12 + 2_000              # one-off data+training amortised + ownership (assumed)

calls_per_day = np.logspace(2, 7, 200)
big_month   = big_per_call * calls_per_day * 30
small_month = small_per_call * calls_per_day * 30 + small_fixed_per_month
crossover = small_fixed_per_month / (30 * (big_per_call - small_per_call))

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(calls_per_day, big_month, label=f"prompt a large model (${big_per_call*1000:.2f} per 1k calls)")
ax.loglog(calls_per_day, small_month, label="fine-tuned small model (per-call + fixed cost)")
ax.axvline(crossover, color="grey", ls="--"); ax.text(crossover * 1.15, 30, f"crossover ≈ {crossover:,.0f} calls/day")
ax.set_xlabel("requests per day"); ax.set_ylabel("cost per month (USD)")
ax.set_title("Total cost of ownership: the crossover is a volume question (illustrative numbers)")
ax.legend(); plt.show()
print(f"large model per call : ${big_per_call:.5f}")
print(f"small model per call : ${small_per_call:.5f}  (+ ${small_fixed_per_month:,.0f}/month fixed)")
print(f"break-even volume    : {crossover:,.0f} requests/day")

Below the crossover, prompting the large model wins; far above it, the small tuned model wins decisively. The term
people forget is the fixed **engineering** cost — an eval suite, a retraining trigger, a rollback plan and an owner.
Latency, data residency and vendor lock-in can move the decision regardless of the dollar line.

## 2. The fine-tuning menu

Two independent choices: **which parameters train** (all of them, or a small add-on) and **what signal they learn
from** (gold answers, preferences, or a reward).

**Which parameters train** (numbers for a Llama-2-7B-shaped model; memory is rough, for training in bf16 with Adam):

| Method | What trains | Trainable params (7B) | GPU memory (7B, rough) | Merge into base? | Use when |
|---|---|---|---|---|---|
| Full fine-tuning | every weight | 6.7B (100%) | ~16 bytes/param → 110 GB+ | n/a | new language/domain, 50k+ clean examples, multi-GPU budget |
| **LoRA** | low-rank A, B beside chosen Linear layers | ~4–40M (0.06–0.6%) | ~16 GB (bf16 base) + small | **yes** — zero added latency | the default for format / style / behaviour |
| **QLoRA** | LoRA on a 4-bit NF4 frozen base | same as LoRA | ~5–8 GB | yes (after de-quantising) | the base does not fit in memory otherwise |
| Bottleneck adapters | small MLPs inserted in each layer | ~30M | like LoRA | no — adds latency | older PEFT; multi-task adapter stacks |
| Prefix tuning | learned key/value prefixes at every layer | ~5M | like LoRA | no — uses context | small capacity, generation tasks |
| Prompt tuning | a few learned "soft token" input embeddings | ~80k | like LoRA | no — uses context | only competitive at very large model scale |
| IA³ | learned scaling vectors on K, V, FFN | ~0.6M | like LoRA | yes | extremely cheap, few-shot regime |

**What signal they learn from** — any of these can be run as full FT or with LoRA:

| Objective | Data per example | Teaches | Notebook |
|---|---|---|---|
| SFT (supervised fine-tuning) | prompt + one gold answer | format, style, the task itself; limited by your ability to *write* good answers | [02](02_sft_from_scratch_with_loss_masking.ipynb), [04](04_lora_with_peft.ipynb) |
| Preference tuning (DPO, RLHF, ORPO, KTO) | prompt + chosen + rejected (or thumbs up/down) | "this is better than that": tone, concision, hedging, refusals | [06](06_dpo_preference_tuning.ipynb) |
| RL on verifiable rewards (PPO, GRPO) | prompt + a reward function (tests pass, answer checks) | reasoning on checkable tasks; needs an automatic scorer | [06](06_dpo_preference_tuning.ipynb) (table) |
| Distillation | prompt + teacher's output (or logits) | a cheaper copy of a bigger model on your task | [07](07_distillation_and_catastrophic_forgetting.ipynb) |

The trainable-parameter column, computed from the layer shapes (32 layers, hidden 4096, MLP 11008):

In [ ]:
L, d, d_ff = 32, 4096, 11008
full = 6_738_415_616                                     # Llama-2-7B parameter count
lora_all = lambda r: L * r * (4 * (d + d) + 2 * (d + d_ff) + (d_ff + d))   # q,k,v,o + gate,up + down
methods = {
    "full fine-tuning":                   full,
    "LoRA r=16, all 7 Linear/layer":      lora_all(16),
    "bottleneck adapters (m=64, 2/layer)": L * 2 * (d * 64 + 64 + 64 * d + d),
    "prefix tuning (20 virtual tokens)":  20 * 2 * L * d,          # a key and a value vector per layer
    "LoRA r=8, q_proj + v_proj only":     L * 2 * 8 * (d + d),
    "IA3 (scale K, V, FFN)":              L * (d + d + d_ff),
    "prompt tuning (20 soft tokens)":     20 * d,
}
for name, n in methods.items():
    print(f"{name:38s} {n:>15,}  ({100 * n / full:8.4f}% of the model)")

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.barh(list(methods)[::-1], list(methods.values())[::-1], color=["#3b5bdb"] * 6 + ["#e8590c"])
ax.set_xscale("log"); ax.set_xlabel("trainable parameters (log scale)")
ax.set_title("Trainable parameters for a 7B model, by fine-tuning method")
plt.show()
assert lora_all(16) / full < 0.01      # LoRA trains well under 1% of the weights

Five orders of magnitude separate full fine-tuning from prompt tuning. LoRA sits in the sweet spot: enough capacity
to change behaviour, small enough to train on one GPU, and — unlike adapters, prefix or prompt tuning — it can be
**merged** back into the base weights for zero extra inference latency. That combination is why LoRA won
(`tn26`). [03 · LoRA from scratch](03_lora_from_scratch.ipynb) derives these counts.

## 3. Data formats

Three shapes cover almost every fine-tuning job:

1. **Chat `messages` JSONL** — one conversation per line; what OpenAI's API and TRL's `SFTTrainer` expect for chat
   models. The last message is the assistant turn the model learns to produce.
2. **Instruction / input / output** ("Alpaca" style) — older, single-turn; convert it to `messages` before training
   a chat model.
3. **Preference pairs** — a prompt plus a *chosen* and a *rejected* answer, for DPO-style tuning.

In [ ]:
chat_example = {"messages": [
    {"role": "system", "content": "You are Acme Cloud's support assistant. Be brief."},
    {"role": "user", "content": "How do I reset my password?"},
    {"role": "assistant", "content": "Go to Settings > Security > Reset password. The link expires in 15 minutes."}]}

alpaca_example = {"instruction": "Classify the ticket into billing, shipping, refund, account or technical.",
                  "input": "I was charged twice for May.", "output": "billing"}

preference_example = {"prompt": "My upload keeps failing. Help?",
                      "chosen": "Sorry about that. Check the file is under 2 GB, then retry from Settings > Uploads.",
                      "rejected": "Uploads fail sometimes. Try again later."}

def alpaca_to_messages(ex: dict, system: str = "You are a helpful assistant.") -> dict:
    user = ex["instruction"] + (("\n\n" + ex["input"]) if ex.get("input") else "")
    return {"messages": [{"role": "system", "content": system}, {"role": "user", "content": user},
                         {"role": "assistant", "content": ex["output"]}]}

work = Path(tempfile.mkdtemp(prefix="ft01_"))
path = work / "train.jsonl"
with path.open("w", encoding="utf-8") as f:
    for ex in [chat_example, alpaca_to_messages(alpaca_example)]:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")          # one JSON object per line, no pretty-printing

lines = path.read_text(encoding="utf-8").splitlines()
print(f"{path.name}: {len(lines)} lines, {path.stat().st_size} bytes")
print(lines[1][:160], "...")
assert all(json.loads(l)["messages"][-1]["role"] == "assistant" for l in lines)
print("\npreference pair keys:", list(preference_example))

JSONL means **one JSON object per line** — no enclosing list, no pretty-printing. A single malformed line fails the
whole upload on hosted APIs, which is why [08 · OpenAI fine-tuning API](08_openai_fine_tuning_api.ipynb) starts with
a validator.

## 4. Data-quality checks on a small dataset

**In plain English:** a thousand carefully checked examples beat fifty thousand scraped ones. Before training, read
the data like a suspicious reviewer: are there duplicates? empty or runaway answers? personal data? a class with
almost no examples? eval rows that also appear in training?

Here is a small support-ticket dataset for a fictional company, with problems planted the way they appear in real
exports.

In [ ]:
rows = [
    ("How do I update my credit card?", "Go to Settings > Billing > Payment methods, add the new card and set it as default.", "billing"),
    ("Why was I charged twice this month?", "A second charge is usually a pending authorisation that drops off within 5 days. If both settle, reply and we will refund one.", "billing"),
    ("Can I get an invoice with our VAT number?", "Yes. Add the VAT number under Settings > Billing > Tax details; new invoices will include it.", "billing"),
    ("Do you offer annual billing?", "Yes. Annual plans are 2 months cheaper; switch under Settings > Billing > Plan.", "billing"),
    ("What payment methods do you accept?", "Visa, Mastercard, Amex and bank transfer for annual plans.", "billing"),
    ("How do I download past invoices?", "Settings > Billing > Invoices lists every invoice with a PDF download button.", "billing"),
    ("Why did my price go up?", "Prices follow your seat count. Check Settings > Billing > Usage for the seat history.", "billing"),
    ("Can I pay by invoice?", "", "billing"),                                                            # empty answer
    ("Is there a discount for nonprofits?", "Yes, 30% off. Email a registration document from the billing page to apply.", "billing"),
    ("How do I change the billing email?", "Settings > Billing > Contacts. Invoices go to the new address from the next cycle.", "billing"),
    ("Hi team, how do I update my credit card?", "Go to Settings > Billing > Payment methods, add the new card and set it as default.", "billing"),  # near-duplicate
    ("Where is my order?", "Track it from Orders > Track shipment; the carrier link updates every few hours.", "shipping"),
    ("Do you ship to Canada?", "Yes, to Canada and the US. Delivery takes 3-7 business days.", "shipping"),
    ("My package arrived damaged.", "Sorry! Send a photo from Orders > Report a problem and we will ship a replacement.", "shipping"),
    ("Can I change my delivery address?", "Only before the order ships: Orders > Edit address.", "shipping"),
    ("How much is express shipping?", "Express is $15 and arrives in 1-2 business days.", "shipping"),
    ("My email is jane.doe@example.com and order 48213 never arrived.", "Sorry about that. We have opened a trace with the carrier and will update you within 24 hours.", "shipping"),  # PII
    ("Do you ship on weekends?", "Orders placed on weekends ship on Monday.", "shipping"),
    ("How do I get a refund?", "Request it from Orders > Refund within 30 days of delivery.", "refund"),
    ("How long does a refund take?", "Refunds reach your card 5-10 business days after we approve them.", "refund"),
    ("Can I get a refund on an annual plan?", "Yes, pro-rated for unused full months.", "refund"),
    ("How do I reset my password?", "Go to Settings > Security > Reset password. The link expires in 15 minutes.", "account"),
    ("How do I reset my password?", "Go to Settings > Security > Reset password. The link expires in 15 minutes.", "account"),   # exact duplicate
    ("how do i reset my password??", "Go to Settings > Security > Reset password. The link expires in 15 minutes.", "account"),  # duplicate after normalising
    ("How do I enable two-factor authentication?", "Settings > Security > Two-factor. Scan the QR code with an authenticator app.", "account"),
    ("How do I delete my account?", "Settings > Account > Delete account. This removes all data after 30 days.", "account"),
    ("Can I add a teammate?", "Yes: Settings > Team > Invite. Each teammate uses one seat.", "account"),
    ("How do I change my username?", "Settings > Profile > Username. You can change it once every 30 days.", "account"),
    ("I am locked out of my account.", "Use Forgot password on the login page; after 5 failed tries wait 15 minutes.", "account"),
    ("The app crashes when I upload a file.", "Update to the latest version and keep files under 2 GB. If it still crashes, send us the log from Help > Diagnostics.", "technical"),
    ("Is there an API?", "Yes. Create a key under Settings > API; the docs are at the developer portal.", "technical"),
    ("Why is sync so slow?", "Large folders sync in the background. Pause other uploads or exclude big folders under Settings > Sync.", "technical"),
    ("Does it work offline?", "Yes. Files marked 'available offline' open without a connection and sync when you reconnect.", "technical"),
    ("Which browsers are supported?", "The latest two versions of Chrome, Firefox, Safari and Edge.", "technical"),
    ("Tell me everything about your security.",
     "Security is very important to us and we take it very seriously. " * 14 + "Contact security@acme.example for details.", "technical"),  # runaway answer
]
df = pd.DataFrame(rows, columns=["prompt", "response", "label"])
print(f"{len(df)} rows; columns: {list(df.columns)}")
print(df["label"].value_counts().to_string())

### 4a. Exact and normalised duplicates

Hash a normalised version of each example (lower-case, collapse whitespace, strip punctuation). Identical hashes are
duplicates. This is O(n) with a dictionary, so it scales to millions of rows.

In [ ]:
def normalise(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df["key"] = [hashlib.sha256(normalise(p + " || " + r).encode()).hexdigest()[:12] for p, r in zip(df.prompt, df.response)]
dupe_mask = df.duplicated("key", keep="first")
print("duplicate rows (after normalising):")
print(df.loc[dupe_mask, ["prompt", "label"]].to_string())
assert dupe_mask.sum() == 2      # the exact copy and the '??' lower-case copy

### 4b. Near-duplicates

Exact hashing misses "Hi team, how do I update my credit card?" — same answer, slightly different question. The
simplest near-duplicate test is **Jaccard similarity** of word sets: |A ∩ B| / |A ∪ B|. At scale you would use
MinHash + locality-sensitive hashing to avoid comparing every pair (`dt09`); with 35 rows we can afford all pairs.

In [ ]:
def jaccard(a: str, b: str) -> float:
    A, B = set(normalise(a).split()), set(normalise(b).split())
    return len(A & B) / max(1, len(A | B))

texts = (df.prompt + " " + df.response).tolist()
pairs = [(i, j, jaccard(texts[i], texts[j])) for i in range(len(texts)) for j in range(i + 1, len(texts))]
near = [(i, j, s) for i, j, s in pairs if s >= 0.8 and df.key[i] != df.key[j]]
for i, j, s in near:
    print(f"rows {i:2d} and {j:2d}  jaccard={s:.2f}\n   {df.prompt[i]!r}\n   {df.prompt[j]!r}")
sims = np.array([s for _, _, s in pairs])
print(f"\n{len(pairs)} pairs compared; median similarity {np.median(sims):.2f}; pairs >= 0.8 that exact hashing missed: {len(near)}")
assert len(near) == 1

### 4c. Empty answers, runaway answers and personal data

Token counts come from the **same tokenizer the model will use**, applied to the example rendered with the model's
chat template — that is what actually occupies the context window during training. We load only the tokenizer of
`HuggingFaceTB/SmolLM2-135M-Instruct` (a few MB, no model weights).

In [ ]:
from transformers import AutoTokenizer

try:
    tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")
except Exception as e:
    raise RuntimeError("Could not load the SmolLM2 tokenizer (needs a one-time download or the HF cache).") from e

SYSTEM = "You are Acme Cloud's support assistant. Be brief."

def to_messages(prompt: str, response: str) -> list[dict]:
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt},
            {"role": "assistant", "content": response}]

def n_tokens(messages) -> int:
    return len(tok.apply_chat_template(messages, tokenize=True, return_dict=True)["input_ids"])

df["tokens"] = [n_tokens(to_messages(p, r)) for p, r in zip(df.prompt, df.response)]
MAX_TOKENS = 128                                  # our training sequence length
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
PHONE = re.compile(r"\+?\d[\d\s().-]{7,}\d")

flags = pd.DataFrame({
    "empty_response": df.response.str.strip() == "",
    "too_long":       df.tokens > MAX_TOKENS,
    "pii":            df.prompt.str.contains(EMAIL) | df.response.str.contains(PHONE),
})
print(df.loc[flags.any(axis=1), ["prompt", "tokens"]].join(flags[flags.any(axis=1)]).to_string())

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
a1.hist(df.tokens, bins=20, color="#3b5bdb")
a1.axvline(MAX_TOKENS, color="#e8590c", ls="--", label=f"max_len = {MAX_TOKENS}")
a1.set_xlabel("tokens per example (chat template applied)"); a1.set_ylabel("examples")
a1.set_title("Token lengths: one runaway example"); a1.legend()
counts = df.label.value_counts()
a2.bar(counts.index, counts.values, color=["#e8590c" if v == counts.min() else "#3b5bdb" for v in counts.values])
a2.set_ylabel("examples"); a2.set_title(f"Label balance: {counts.idxmax()} has {counts.max() / counts.min():.1f}x {counts.idxmin()}")
plt.tight_layout(); plt.show()

What to do with each finding:

| Finding | Why it hurts | Fix |
|---|---|---|
| Duplicates / near-duplicates | over-weights those rows; leaks into the eval split | drop them *before* splitting |
| Empty answer | teaches the model that saying nothing is fine | drop (or write the answer) |
| Runaway answer | teaches rambling; gets truncated mid-sentence at `max_len` | fix or drop; never silently truncate |
| Personal data | the model can memorise and repeat it | redact at ingestion (`[EMAIL]`), not later |
| Rare class (`refund`) | the model gets better only at the common intents | collect or up-sample the rare class |

In [ ]:
clean = df.loc[~dupe_mask & ~flags.empty_response & ~flags.too_long].copy()
clean = clean.drop(index=[j for _, j, _ in near])                       # keep the first of each near-duplicate pair
clean["prompt"] = clean.prompt.str.replace(EMAIL, "[EMAIL]", regex=True)
print(f"rows: {len(df)} raw -> {len(clean)} clean")
print("redacted:", clean.loc[clean.prompt.str.contains(r"\[EMAIL\]"), "prompt"].iloc[0])
assert not clean.prompt.str.contains(EMAIL).any()

### 4d. Train / validation split — and leakage

**Leakage** means the same (or nearly the same) example sits in both train and eval, so the eval score measures
memorisation. The classic mistake is to split *before* de-duplicating. Let us measure it: split the **raw** data
at random, then count validation rows that have a near-twin in training.

In [ ]:
from sklearn.model_selection import train_test_split

def leaked_rows(train: pd.DataFrame, val: pd.DataFrame, threshold: float = 0.8) -> int:
    tr = (train.prompt + " " + train.response).tolist()
    return sum(any(jaccard(v, t) >= threshold for t in tr) for v in (val.prompt + " " + val.response))

naive_train, naive_val = train_test_split(df, test_size=0.3, random_state=3)
good_train, good_val = train_test_split(clean, test_size=0.3, random_state=0, stratify=clean.label)
print(f"naive split (raw data)    : {leaked_rows(naive_train, naive_val)} of {len(naive_val)} val rows leak")
print(f"clean split (dedupe first): {leaked_rows(good_train, good_val)} of {len(good_val)} val rows leak")
print("\nclean val label counts:", good_val.label.value_counts().to_dict())
assert leaked_rows(good_train, good_val) == 0 and leaked_rows(naive_train, naive_val) > 0

De-duplicate first, then split — and `stratify=` keeps every label represented in validation. For data from
**production logs**, split **by time** instead of at random (train on older traffic, hold out the most recent
slice): a random split leaks the future distribution into training and inflates the eval (`tn25`). And create the
held-out split **before** you generate or augment any examples.

## 5. Chat templates — the silent bug

**In plain English:** an instruct model was trained with invisible markers that say who is talking. If your training
data (or your inference code) uses different markers, it still "works" — just worse, and nothing tells you.

`tokenizer.apply_chat_template` renders messages exactly the way the model was trained. Here is SmolLM2's format:

In [ ]:
msgs = to_messages("How do I reset my password?", "Settings > Security > Reset password.")
rendered = tok.apply_chat_template(msgs, tokenize=False)
print(rendered)
print("special tokens:", {"bos": tok.bos_token, "eos": tok.eos_token, "pad": tok.pad_token})
print("ids           :", {t: tok.convert_tokens_to_ids(t) for t in ["<|im_start|>", "<|im_end|>"]})

SmolLM2 uses the **ChatML** layout: each turn is `<|im_start|>role\n ... <|im_end|>`. Three details matter when you
train:

1. `<|im_end|>` is the **EOS** token. The model learns to *stop* only if `<|im_end|>` is inside the part the loss
   is computed on ([02](02_sft_from_scratch_with_loss_masking.ipynb) shows what happens otherwise).
2. The **pad** token is *also* `<|im_end|>`. If a collator masks "every pad token" out of the loss, it also masks
   the real EOS — and the model never learns to stop. Mask by position (attention mask), not by token id.
3. The template has hidden behaviour. Watch what it does when there is no system message:

In [ ]:
no_system = tok.apply_chat_template([{"role": "user", "content": "Hi"}], tokenize=False, add_generation_prompt=True)
print(no_system)
assert "SmolLM" in no_system       # a default system prompt was injected

With no system message, SmolLM2's template silently **injects its own default system prompt**, and
`add_generation_prompt=True` appends `<|im_start|>assistant\n` so the model knows it is its turn. Train and serve
with the *same* system prompt and the *same* template, or you train one distribution and serve another.

Now the token view — what the model actually sees. The table shows the tokens of the prompt part and of the answer
part; only the answer tokens (plus the closing `<|im_end|>`) should be learned.

In [ ]:
full_ids = tok.apply_chat_template(msgs, tokenize=True, return_dict=True)["input_ids"]
prompt_ids = tok.apply_chat_template(msgs[:-1], tokenize=True, return_dict=True, add_generation_prompt=True)["input_ids"]
assert full_ids[:len(prompt_ids)] == prompt_ids         # the prompt is an exact prefix of the full example

pieces = tok.convert_ids_to_tokens(full_ids)
view = pd.DataFrame({"id": full_ids, "token": [p.replace("Ċ", "\\n").replace("Ġ", "·") for p in pieces],
                     "part": ["prompt"] * len(prompt_ids) + ["answer"] * (len(full_ids) - len(prompt_ids))})
view["special"] = view.id.isin(tok.all_special_ids)
print(f"{len(full_ids)} tokens: {len(prompt_ids)} prompt + {len(full_ids) - len(prompt_ids)} answer")
print(view.iloc[len(prompt_ids) - 4:].to_string())

(`Ġ` marks a leading space and `Ċ` a newline in this byte-level BPE vocabulary; the table shows them as `·` and
`\n`.) Some templates can hand you this mask directly with `return_assistant_tokens_mask=True`, but only if the
template marks assistant text with a `{% generation %}` tag. SmolLM2's does not — check before you rely on it:

In [ ]:
out = tok.apply_chat_template(msgs, tokenize=True, return_dict=True, return_assistant_tokens_mask=True)
print("assistant tokens found by the template:", sum(out["assistant_masks"]))
print("'{% generation %}' in template        :", "{% generation %}" in tok.chat_template)

Zero — the flag silently returns an all-zero mask for this model. That is why SFT code usually builds the mask the
way we just did: tokenize the prompt with `add_generation_prompt=True`, and mask that many leading tokens.

The mismatch bug, in numbers: a hand-rolled `"User: ... Assistant: ..."` string and the real template produce
different token sequences for the same conversation.

In [ ]:
hand_rolled = f"System: {SYSTEM}\nUser: How do I reset my password?\nAssistant: Settings > Security > Reset password."
hand_ids = tok(hand_rolled)["input_ids"]
shared = len(set(hand_ids) & set(full_ids)) / len(set(full_ids))
print(f"template tokens: {len(full_ids)}, hand-rolled tokens: {len(hand_ids)}, "
      f"special tokens in hand-rolled: {sum(i in tok.all_special_ids for i in hand_ids)}")
print(f"share of the template's distinct tokens that also appear hand-rolled: {shared:.0%}")

The hand-rolled string contains **no** special tokens: the model never sees the turn markers it was trained on.
It still produces an answer — which is exactly why this bug survives code review.

## 6. How many examples?

Far fewer than people expect, and quality beats quantity. These ranges are **rough rules of thumb** from practice
and papers (LIMA: 1,000 curated examples made a strong assistant); your task can differ by an order of magnitude.

In [ ]:
ranges = [  # (goal, low, high)
    ("OpenAI minimum / first signal (their docs: 10 min., gains from ~50-100)", 10, 100),
    ("format or style change (LoRA)", 100, 500),
    ("domain behaviour (LoRA)", 1_000, 10_000),
    ("preference pairs for DPO (a common start)", 1_000, 10_000),
    ("general instruction tuning (SFT)", 10_000, 1_000_000),
    ("new capability / language (full FT)", 50_000, 1_000_000),
]
fig, ax = plt.subplots(figsize=(10, 3.4))
for k, (goal, lo, hi) in enumerate(ranges[::-1]):
    ax.barh(k, np.log10(hi) - np.log10(lo), left=np.log10(lo), color="#3b5bdb", alpha=0.8)
    ax.text(np.log10(hi) + 0.05, k, f"{lo:,}–{hi:,}", va="center", fontsize=8)
ax.axvline(3, color="#e8590c", ls="--"); ax.text(3.03, len(ranges) - 0.6, "LIMA: 1,000 curated", color="#e8590c", fontsize=8)
ax.set_yticks(range(len(ranges))); ax.set_yticklabels([g for g, _, _ in ranges[::-1]], fontsize=8)
ax.set_xticks(range(1, 7)); ax.set_xticklabels([f"{10**i:,}" for i in range(1, 7)])
ax.set_xlim(0.8, 7); ax.set_xlabel("examples (log scale)"); ax.set_title("How many examples? Rough, hedged ranges")
plt.tight_layout(); plt.show()

What "good" looks like, in the order interviewers ask about it (`tn13`): real input distribution (not prompts you
invented), consistent formatting, correct and complete outputs, diverse coverage, de-duplicated, **negative and edge
cases** (refusals, ambiguous and malformed inputs — or the model will handle none of them), and a held-out test split
created *before* any training.

## 7. The pre-flight checklist

Run this before you spend a GPU-hour. Each check is one line of code on the cleaned data.

In [ ]:
def preflight(train: pd.DataFrame, val: pd.DataFrame) -> pd.DataFrame:
    tr_keys, va_keys = set(train.key), set(val.key)
    counts = train.label.value_counts()
    checks = [
        ("no duplicate rows in train",         not train.duplicated("key").any()),
        ("no empty answers",                   (train.response.str.strip() != "").all()),
        (f"all examples <= {MAX_TOKENS} tokens", (train.tokens <= MAX_TOKENS).all()),
        ("no raw e-mail addresses (PII)",      not train.prompt.str.contains(EMAIL).any()),
        ("val shares no rows with train",      not (tr_keys & va_keys)),
        ("no near-duplicate leakage",          leaked_rows(train, val) == 0),
        ("every label present in val",         set(val.label) == set(train.label)),
        ("label imbalance <= 5x",              counts.max() / counts.min() <= 5),
        ("enough data for a LoRA style tune (>= 100)", len(train) >= 100),
    ]
    return pd.DataFrame(checks, columns=["check", "ok"]).assign(ok=lambda d: d.ok.map({True: "PASS", False: "WARN"}))

report = preflight(good_train, good_val)
print(report.to_string(index=False))
assert (report.ok == "PASS").sum() >= 7

The last check warns, correctly: this toy set is far too small for a real fine-tune — its job was to show the checks.
Things the code cannot check, but you must: the chat template and system prompt at training equal the ones at
inference; the loss covers only assistant tokens (and includes EOS); and you measured the **base model** on the
validation set first, so you have a number to beat.

## Try it yourself

**1.** Add a scenario to `recommend`: *"The model must return valid JSON for our schema 100% of the time."* Which rung
does it land on, and what would you try before fine-tuning? (Hint: structured outputs / constrained decoding.)

In [ ]:
# Solution — try it yourself first, then run this
s = dict(prompt_tried=True, behaviour_wrong=True, n_examples=300)
print(recommend(s))
print("Before tuning: use the API's structured outputs (JSON schema) or constrained decoding — that guarantees the"
      " format at decode time; tune only if the *content* inside the JSON is still wrong.")

**2.** Print the five most similar pairs that exact hashing did not already catch, then try thresholds 0.8, 0.5,
0.3 and 0.25. Where is the gap between "real near-duplicate" and "different question", and what gets flagged
below it?

In [ ]:
# Solution — try it yourself first, then run this
candidates = sorted([(s, i, j) for i, j, s in pairs if df.key[i] != df.key[j]], reverse=True)
for s, i, j in candidates[:5]:
    print(f"{s:.2f}  {df.prompt[i][:38]!r:42s} vs {df.prompt[j][:38]!r}")
for thr in (0.8, 0.5, 0.3, 0.25):
    print(f"threshold {thr}: {sum(s >= thr for s, _, _ in candidates)} pairs flagged")

The one true near-duplicate scores about 0.9; the next pairs score about 0.3 and are *different* questions that
share boilerplate ("How do I ...", "Settings > ..."). Put the threshold in that gap — and choose it by reading a
sample of flagged pairs, not by guessing.

**3.** Render a **two-turn** conversation (user, assistant, user, assistant) with the template. How many tokens are
assistant tokens that a correct SFT mask would train on?

In [ ]:
# Solution — try it yourself first, then run this
conv = [{"role": "system", "content": SYSTEM},
        {"role": "user", "content": "Do you ship to Canada?"}, {"role": "assistant", "content": "Yes, in 3-7 days."},
        {"role": "user", "content": "And express?"}, {"role": "assistant", "content": "Express is $15, 1-2 days."}]
ids = tok.apply_chat_template(conv, tokenize=True, return_dict=True)["input_ids"]
mask = [0] * len(ids)
for k, m in enumerate(conv):
    if m["role"] == "assistant":                       # tokens between this turn's prompt prefix and its end
        start = len(tok.apply_chat_template(conv[:k], tokenize=True, return_dict=True, add_generation_prompt=True)["input_ids"])
        end = len(tok.apply_chat_template(conv[:k + 1], tokenize=True, return_dict=True)["input_ids"])
        mask[start:end] = [1] * (end - start)
print(f"{len(ids)} tokens, {sum(mask)} assistant tokens (both answers, each with its <|im_end|> and newline)")
print(repr(tok.decode([i for i, m in zip(ids, mask) if m])))

**4.** Convert `preference_example` into the shape OpenAI's DPO fine-tuning expects: `input.messages`,
`preferred_output` and `non_preferred_output` (each a list with one assistant message).

In [ ]:
# Solution — try it yourself first, then run this
openai_dpo = {"input": {"messages": [{"role": "user", "content": preference_example["prompt"]}]},
              "preferred_output": [{"role": "assistant", "content": preference_example["chosen"]}],
              "non_preferred_output": [{"role": "assistant", "content": preference_example["rejected"]}]}
print(json.dumps(openai_dpo, indent=1)[:400])

## Say it in an interview

- **30-second answer to "should we fine-tune?":** "First I'd check it's not a prompting problem — instructions plus
  two examples. If the model lacks or needs fresh, citable facts, that's retrieval: weights can't be updated weekly,
  cited, permission-filtered or have a fact deleted. Fine-tuning is for consistent *behaviour* — format, tone, a
  narrow task — once I have a few hundred good examples and a baseline eval. They compose: tuned for format, fed by
  retrieval."
- **Menu:** full FT (all weights, ~16 bytes/param, 110 GB+ for 7B) vs LoRA (<1% of weights, mergeable, the default)
  vs QLoRA (LoRA on a 4-bit base when it does not fit). Objectives: SFT (gold answers) → preference tuning (chosen vs
  rejected) → RL on verifiable rewards.
- **Data:** quality over quantity (LIMA: 1,000 curated). 100–500 examples for a format change, 1k–10k for domain
  behaviour, 50k+ for a new language. De-duplicate *before* splitting; split production logs by time; redact PII at
  ingestion; include refusals and edge cases.
- **Chat template:** always `apply_chat_template`, same template and system prompt at train and serve time, loss on
  assistant tokens only, EOS inside the loss. Watch for pad == eos (SmolLM2, Llama-3 recipes) and templates that
  inject a default system prompt.
- **Traps:** "we fine-tuned on our docs so it knows them" (anti-pattern — name it first); an eval set generated or
  split *after* augmentation (leaks); forgetting the fixed engineering cost when comparing a tuned small model with a
  prompted large one.

## Next

- [02 · SFT from scratch with loss masking](02_sft_from_scratch_with_loss_masking.ipynb) — the mask, EOS and packing, trained for real
- [03 · LoRA from scratch](03_lora_from_scratch.ipynb) · [04 · LoRA with PEFT](04_lora_with_peft.ipynb)
- [08 · OpenAI fine-tuning API](08_openai_fine_tuning_api.ipynb) — the hosted route, with a JSONL validator
- [RAG pipeline from scratch](../09_rag/06_rag_pipeline_from_scratch.ipynb) — the alternative for knowledge problems
- [Tokenizers, models and generate](../10_huggingface/02_tokenizers_models_and_generate.ipynb) · [Datasets library](../10_huggingface/03_datasets_library.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)